In [18]:
import numpy as np
import torch
from math import e

In [19]:
# 입력값
X = np.array([160, 170, 180, 190])
# 정답
y = np.array([0, 0, 1, 1])

X, y

(array([160, 170, 180, 190]), array([0, 0, 1, 1]))

In [20]:
# 입력값 정규화
X_mean = np.mean(X)
X_std = np.std(X)

X_norm = (X - X_mean) / X_std


In [21]:
# tensor로 변환
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

X_norm_tensor, y_tensor

(tensor([-1.3416, -0.4472,  0.4472,  1.3416]), tensor([0., 0., 1., 1.]))

In [22]:
# 가중치 편향 초기값 설정
a = torch.tensor(0.1, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

a.item(), b.item()

(0.10000000149011612, 0.0)

In [23]:
# 학습 설정
learning_rate = 0.1
epochs = 1000

# optimizer 생성
# [a, b]: 학습 대상(파라미터) 지정
# optimizer 로 grad값 0초기화, a b 업데이트 모두 수행
optimizer = torch.optim.SGD([a, b], lr=learning_rate)

optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [24]:
def sigmoid(H):
    return 1 / (1+ e **(-H))



In [25]:
# 학습 전 예측 확인
H = a.item() * X_norm + b.item()
z = sigmoid(H)

H, z

(array([-0.13416408, -0.04472136,  0.04472136,  0.13416408]),
 array([0.4665092 , 0.48882152, 0.51117848, 0.5334908 ]))

In [26]:
# cost 계산
# log(0) 방지

epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1 - epsilon)

costs = - (y * np.log(z_safe) + (1-y) * np.log(1-z_safe))
Cost = np.mean(costs)
costs, Cost

(array([0.62831345, 0.67103648, 0.67103648, 0.62831345]),
 np.float64(0.6496749672265923))

In [ ]:
# optimizer로 학습
for epoch in range(epochs):
    # grad초기화
    optimizer.zero_grad()

    H = a * X_norm_tensor + b
    z = torch.sigmoid(H)
    z_safe = torch.clamp(z, epsilon, 1 - epsilon)

    costs = -(
        y_tensor * torch.log(z_safe) + (1 - y_tensor) * torch.log(1 - z_safe)
    )
    mean_cost = torch.mean(costs)

    # a.grad, b.grad계산
    mean_cost.backward()
    # optimizer가 a, b업데이트
    optimizer.step()

    if epoch%100 == 0 or epoch == epochs - 1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}'
            f'a={a.item():.6f}'
            f'b={b.item():.6f}'
        )
    if epoch < 3:
        grad_a = a.grad.item()
        grad_b = b.grad.item()
        print(
            f'  (확인용) a.grad={grad_a:.6f}'
            f'b.grad={grad_b:.6f}'
        )

  (확인용) a.grad=-0.422248b.grad=0.000000
  (확인용) a.grad=-0.411755b.grad=0.000000
  (확인용) a.grad=-0.401573b.grad=0.000000
epoch=9, Cost=0.518612a=0.478745b=0.000000
epoch=109, Cost=0.189122a=2.159186b=0.000000
epoch=209, Cost=0.130317a=2.916380b=0.000000
epoch=309, Cost=0.102238a=3.443422b=0.000000
epoch=409, Cost=0.084896a=3.858503b=0.000000
epoch=509, Cost=0.072851a=4.204756b=0.000000
epoch=609, Cost=0.063901a=4.503386b=0.000000
epoch=709, Cost=0.056950a=4.766657b=0.000000
epoch=809, Cost=0.051378a=5.002427b=0.000000
epoch=909, Cost=0.046803a=5.216084b=0.000000
epoch=999, Cost=0.043330a=5.392711b=0.000000


In [28]:
print('학습된 a:', a.item())
print('학습된 b:', b.item())


학습된 a: 5.3927106857299805
학습된 b: 8.616920865733846e-08


In [29]:
# 새로운 입력값 예측
input_height = 185
input_norm = (input_height - X_mean) / X_std

with torch.no_grad():
    input_norm_tensor = torch.tensor(input_norm, dtype=torch.float32)
    H_new = a * input_norm_tensor + b
    z_new = torch.sigmoid(H_new)
    pred = 1 if z_new.item() >= 0.5 else 0

print(f'키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}')
if pred == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

키가 185cm인 사람의 농구선수 확률(z): 0.9920
판별 결과: 농구선수입니다.
